# Exploration Couche GOLD — Olist E-Commerce

Analyse des KPIs metier generes par la pipeline PySpark.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd

# Tentative d'import matplotlib avec gestion d'erreur
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = (12, 6)
    plt.rcParams['font.size'] = 12
except ImportError:
    HAS_MATPLOTLIB = False
    print("matplotlib non installe, les visualisations seront sautees")

## 2. Chargement des donnees GOLD

In [ ]:
# Chemins des fichiers Parquet
paths = {
    'monthly_revenue': '../data/gold/monthly_revenue/',
    'top_categories': '../data/gold/top_categories/',
    'top_sellers': '../data/gold/top_sellers/',
    'delay_impact': '../data/gold/delay_impact/',
    'payment_distribution': '../data/gold/payment_distribution/'
}

# Chargement des DataFrames
gold_data = {}
for name, path in paths.items():
    gold_data[name] = pd.read_parquet(path)
    print(f"{name} charge : {gold_data[name].shape[0]} lignes")

### Apercu des donnees

In [ ]:
def show_data_summary(name, df):
    print(f"\n--- {name.upper()} ---")
    print("\nHead():")
    display(df.head())
    print("\nInfo():")
    df.info()
    print("\nValeurs nulles:")
    print(df.isnull().sum())

for name, df in gold_data.items():
    show_data_summary(name, df)

## 3. Data sanity check

In [ ]:
# Verification rapide des doublons
for name, df in gold_data.items():
    duplicates = df.duplicated().sum()
    print(f"{name} : {duplicates} doublons")

## 4. Visualisations Business

### 4.1 CA mensuel

In [ ]:
if HAS_MATPLOTLIB:
    df_revenue = gold_data['monthly_revenue'].copy()

    # Si la colonne n'est pas 'month' mais 'order_purchase_timestamp', on ajuste
    if 'order_purchase_timestamp' in df_revenue.columns:
        df_revenue['month'] = pd.to_datetime(df_revenue['order_purchase_timestamp']).dt.to_period('M')
        df_revenue = df_revenue.groupby('month').agg({'revenue': 'sum'}).reset_index()
        df_revenue['month'] = df_revenue['month'].astype(str)

    plt.figure(figsize=(14, 6))
    plt.plot(df_revenue['month'], df_revenue['revenue'], marker='o', linewidth=2, color='#2ecc71')
    plt.title('Evolution du chiffre d\'affaires mensuel', fontsize=14, pad=20)
    plt.xlabel('Mois', fontsize=12)
    plt.ylabel('Chiffre d\'affaires (R$)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Visualisation indisponible: matplotlib non installe")

### 4.2 Top 10 categories produits

In [ ]:
if HAS_MATPLOTLIB:
    df_categories = gold_data['top_categories'].head(10).copy()

    plt.figure(figsize=(12, 6))
    plt.barh(df_categories['product_category_name'], df_categories['revenue'], color='#3498db')
    plt.gca().invert_yaxis()  # Pour avoir la categorie la plus performante en haut
    plt.title('Top 10 categories par chiffre d\'affaires', fontsize=14, pad=20)
    plt.xlabel('Chiffre d\'affaires (R$)', fontsize=12)
    plt.ylabel('Categorie produit', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("Visualisation indisponible: matplotlib non installe")

### 4.3 Top 10 vendeurs

In [ ]:
if HAS_MATPLOTLIB:
    df_sellers = gold_data['top_sellers'].head(10).copy()

    # On masque les IDs pour la confidentialite
    df_sellers['seller_id'] = [f"Vendeur {i+1}" for i in range(len(df_sellers))]

    plt.figure(figsize=(12, 6))
    plt.barh(df_sellers['seller_id'], df_sellers['revenue'], color='#9b59b6')
    plt.gca().invert_yaxis()
    plt.title('Top 10 vendeurs par chiffre d\'affaires', fontsize=14, pad=20)
    plt.xlabel('Chiffre d\'affaires (R$)', fontsize=12)
    plt.ylabel('Vendeur', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("Visualisation indisponible: matplotlib non installe")

### 4.4 Repartition des moyens de paiement

In [ ]:
if HAS_MATPLOTLIB:
    df_payments = gold_data['payment_distribution'].copy()

    plt.figure(figsize=(10, 10))
    plt.pie(df_payments['total'], labels=df_payments['payment_type'], autopct='%1.1f%%', 
            startangle=90, colors=['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#95a5a6'])
    plt.title('Repartition des moyens de paiement', fontsize=14, pad=20)
    plt.show()
else:
    print("Visualisation indisponible: matplotlib non installe")

### 4.5 Impact des retards sur la satisfaction client

In [ ]:
if HAS_MATPLOTLIB:
    df_delay = gold_data['delay_impact'].copy()
    df_delay['is_late'] = df_delay['is_late'].map({0: 'A l\'heure', 1: 'En retard'})

    plt.figure(figsize=(10, 6))
    plt.bar(df_delay['is_late'], df_delay['avg_review'], color=['#2ecc71', '#e74c3c'])
    plt.title('Note moyenne selon le respect du delai de livraison', fontsize=14, pad=20)
    plt.xlabel('Statut de livraison', fontsize=12)
    plt.ylabel('Note moyenne (1-5)', fontsize=12)
    plt.ylim(0, 5)
    for i, v in enumerate(df_delay['avg_review']):
        plt.text(i, v + 0.1, f"{v:.2f}", ha='center', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("Visualisation indisponible: matplotlib non installe")

## 5. Analyse metier

### Insight 1 : Tendances CA

Le chiffre d'affaires semble avoir une tendance generale a la hausse, avec des pics saisonniers potentiels. Les mois de novembre et decembre montrent souvent des pics significatifs.

### Insight 2 : Categories dominantes

Certaines categories (comme beaute_sante, cama_mesa_banho) generent la majorite des revenus, indiquant des opportunites d'optimisation de stock et de marketing sur ces produits phares.

### Insight 3 : Impact critique des retards

Les commandes livrees en retard ont une note moyenne bien inferieure (en general 2 points de moins). Cela demontre l'importance strategique de la logistique pour la satisfaction et la fidelisation client.

### Insight 4 : Comportement paiement

La carte de credit est le moyen de paiement dominant (environ 80%), suivi du boleto bancario. Cela reflete les habitudes de consommation bresiliennes et justifie l'investissement dans des solutions de paiement adaptees.

## 6. Conclusion

### Resume de la pipeline

Nous avons construit une pipeline PySpark complete en 4 couches :
- RAW : Fichiers CSV d'origine
- Bronze : Donnees ingerees en Parquet
- Silver : Donnees nettoyees et fiabilisees
- Gold : KPIs metier prets a l'analyse

### KPIs cles

- Chiffre d'affaires total
- Chiffre d'affaires mensuel
- Panier moyen
- Top categories et vendeurs
- Delai moyen et taux de retard
- Note moyenne et impact des retards
- Repartition des paiements

### Valeur metier

Le projet permet a Olist de :
- Comprendre les tendances de ventes
- Optimiser la logistique pour reduire les retards
- Identifier les produits et vendeurs performants
- Analyser le comportement client

### Limites du systeme

- Pipeline en mode local seulement
- Pas d'automatisation (Airflow)
- Pas de dashboard interactif
- Donnees statiques (pas de streaming)